# L4 Taxonomy Classification (Anchor-Based)

Classifies product and service listings into the 13 unified L4 taxonomy categories using
cosine similarity between listing embeddings and taxonomy anchor embeddings.

**Approach:** Embed L4 anchor descriptions → compute cosine similarity against cached listing
embeddings → assign highest-scoring L4 → flag low-confidence items for residual clustering.

**Modes:**
- `validate_products`: Load from LEI products table. Validates anchor descriptions by checking
  whether known products land on expected L4s. Use this to calibrate `CONFIDENCE_THRESHOLD`
  before classifying services.
- `classify_services`: Load services product buckets from the SERVICES table. Runs the full
  classification pipeline.

**Prerequisites:**
- `analysis/data/l4_taxonomy_anchors.json` — L4 anchor definitions (finalize descriptions before running)
- Titan embedding cache populated for the target dataset (LEI run must complete before services run)


In [1]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    raise FileNotFoundError("Could not locate project root containing product_classifier_utils.py")

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import (
    build_product_text,
    embed_texts_from_cache,
    get_bedrock_client,
    get_snowflake_session,
    stable_text_hash,
)

In [2]:
# ── Run mode ─────────────────────────────────────────────────────────────────
# "validate_products" : Load LEI products table. Use for threshold calibration.
# "classify_services" : Load services product buckets for full classification.
MODE = "classify_services"

# ── Snowflake tables ──────────────────────────────────────────────────────────
PRODUCTS_TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_LEI"
SERVICES_TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.SERVICES"

# Which service buckets to include in classify_services mode.
# These are the product-heavy buckets identified from the data analysis.
# Set to None to load all services (not recommended — includes genuine services).
SERVICE_BUCKETS = [
    # (PARENT_3_CATEGORY, PARENT_4_CATEGORY or None for blank)
    ("Chemistry and Materials", None),          # 4.84M blank-L4 chemicals
    ("Compounds", None),                        # 747K raw compounds
    ("Biology", "Cells and Tissues"),           # 570K misclassified products
    ("Biology", "Biochemistry & Molecular Biology"),  # 2.93M ambiguous
]

# Optionally limit rows for development / testing
ROW_LIMIT = None  # set to e.g. 10_000 for a quick test run

# ── Embedding + AWS ───────────────────────────────────────────────────────────
AWS_PROFILE = "staging.admin"
AWS_REGION = "us-east-1"
MODEL_ID = "amazon.titan-embed-text-v1"
MAX_WORKERS = 16
CHECKPOINT_EVERY = 50000  # Now safe — checkpoints only write new embeddings, not full cache
MAX_RETRIES = 5
CACHE_PATH = PROJECT_ROOT / "artifacts/cache/embedding_cache.pkl"

# ── Classification ────────────────────────────────────────────────────────────
ANCHORS_PATH = PROJECT_ROOT / "analysis/data/l4_taxonomy_anchors.json"

# Confidence threshold below which a listing is flagged for residual analysis.
# Cosine similarity range is [-1, 1]; typical well-matched listings score > 0.65.
# Calibrate against validate_products run before applying to services.
CONFIDENCE_THRESHOLD = 0.50  # Titan embeddings typically score 0.45–0.65; calibrate after seeing the histogram

# Whether to embed texts missing from cache. Set False to do a dry-run that
# reports cache coverage without calling Titan (useful for scoping overnight jobs).
RUN_EMBEDDING = True

# ── Output ────────────────────────────────────────────────────────────────────
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "analysis"
RUN_TAG = f"l4_classification_{MODE}"
SAVE_TO_SNOWFLAKE = False
OUTPUT_TABLE = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.L4_CLASSIFICATIONS"

## 1. Load L4 Taxonomy Anchors

In [3]:
with open(ANCHORS_PATH) as f:
    anchors_data = json.load(f)

anchors = anchors_data["anchors"]
anchor_ids    = [a["id"]          for a in anchors]
anchor_labels = [a["label"]       for a in anchors]
anchor_texts  = [a["description"] for a in anchors]

print(f"Loaded {len(anchors)} L4 taxonomy anchors:")
for i, (aid, label) in enumerate(zip(anchor_ids, anchor_labels)):
    print(f"  {i:2d}. [{aid}] {label}")

Loaded 13 L4 taxonomy anchors:
   0. [chemicals_solvents] Chemicals & Solvents
   1. [molecular_biology_reagents] Molecular Biology Reagents
   2. [lab_supplies_consumables] Lab Supplies & Consumables
   3. [antibodies] Antibodies
   4. [proteins_peptides] Proteins & Peptides
   5. [kits_assays] Kits & Assays
   6. [cell_tissue_culture] Cell & Tissue Culture
   7. [biospecimens] Biospecimens
   8. [animal_models] Animal Models
   9. [equipment_instruments_parts] Equipment, Instruments & Parts
  10. [controls_calibrators_standards] Controls, Calibrators & Standards
  11. [furniture_storage] Furniture & Storage
  12. [general_office_supplies] General Office Supplies


## 2. Connect to Snowflake and Load Listings

In [4]:
sf_session = get_snowflake_session()

def _build_services_where(buckets):
    """Build WHERE clause for the service product bucket filters."""
    clauses = []
    for l3, l4 in buckets:
        if l4 is None:
            clauses.append(
                f"(PARENT_3_CATEGORY = '{l3}' AND "
                f"(PARENT_4_CATEGORY IS NULL OR TRIM(PARENT_4_CATEGORY) = ''))"
            )
        else:
            clauses.append(
                f"(PARENT_3_CATEGORY = '{l3}' AND PARENT_4_CATEGORY = '{l4}')"
            )
    return "(\n  " + "\n  OR ".join(clauses) + "\n)"


if MODE == "validate_products":
    query = f"""
    SELECT
        PRODUCT_ID,
        PRODUCT_NAME,
        DESCRIPTION,
        PRICING_STATUS_C,
        LIST_PRICE_C,
        PARENT_3_CATEGORY AS CURRENT_L3
    FROM {PRODUCTS_TABLE}
    WHERE UPPER(COALESCE(DESCRIPTION, '')) NOT LIKE '%INSERT%'
    """
    if ROW_LIMIT:
        query += f"\nLIMIT {int(ROW_LIMIT)}"

elif MODE == "classify_services":
    # SERVICES table is pre-filtered upstream in Snowflake to the 7M priced listings of interest.
    # SERVICE_BUCKETS is kept for reference but no WHERE clause is applied.
    query = f"""
    SELECT
        PRODUCT_ID,
        PRODUCT_NAME,
        DESCRIPTION,
        PRICING_STATUS_C,
        LIST_PRICE_C,
        PARENT_3_CATEGORY AS CURRENT_L3,
        PARENT_4_CATEGORY AS CURRENT_L4,
        PARENT_5_CATEGORY AS CURRENT_L5
    FROM {SERVICES_TABLE}
    """
    if ROW_LIMIT:
        query += f"\nLIMIT {int(ROW_LIMIT)}"

else:
    raise ValueError(f"Unknown MODE: {MODE!r}. Must be 'validate_products' or 'classify_services'.")

print(f"Query (MODE={MODE}):")
print(query)

df = sf_session.sql(query).to_pandas()
print(f"\nLoaded rows: {len(df):,}")
print(df.head(3))

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://scienceexchange.okta.com/app/snowflake/exktzixqmxM7e7kdB697/sso/saml?SAMLRequest=lZJfb9owFMW%2FSuQ9J3Zo0xQLqCisW6ayIaCT1jeTXMBLYqe%2BDkn59HP4M3UPrbS3yDnHv%2BN77uCuLQtvDwalVkMSBox4oFKdSbUdkqfVg39LPLRCZaLQCobkFZDcjQYoyqLi49ru1AJeakDruYsU8u7HkNRGcS1QIleiBOQ25cvx7JH3AsYFIhjrcORsyVA61s7ailPaNE3QXAXabGmPMUZZnzpVJ%2FlE3iCqjxmV0VanurhYWvemdxAhZdcdwikcYX423kt1GsFHlPVJhPzrajX35z%2BWK%2BKNL6%2BbaIV1CWYJZi9TeFo8ngKgS1DlaRRfR7dBjT4ItH4YoNLNphA5pLqsauuuDdwX3UBGC72VbljJdEiqXGbNfH27j5K90b%2FW4jn%2B0oPiNyRs1hw%2Bf9P5RImFmLSHtdIsT4n381Jtr6s2QawhUV2h1h2x3o3PIj%2BMVuyKs5CzKAjj%2FjPxpq5QqYQ9Oi%2BpMZVuSABtuhNqC4HOrTiGFFVF%2F%2Ban0Ob2INuXsp3FEOfZ%2FU0%2Fpoiadr2R0%2BrwYxAz%2Bu%2BBDOhb%2B3kNv7tmkulcFzJ99R60KYV9v7gwCI8nMvM3RymHUshinGUGEF2BRaGbiQFh3bZbUwOhoxP1330f%2FQE%3D&RelayState=ver%3A3-hint%3A9376132683223046-ETMsDgAAAZ4plJobABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEIDVK685esFWY984mcOfaeIAAACgZQ5Dt

## 3. Load Embedding Cache

In [5]:
def save_cache(cache: dict, path: Path) -> None:
    """Atomic cache write: write to temp file then rename to avoid corruption on interrupt."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix('.pkl.tmp')
    with open(tmp_path, "wb") as f:
        pickle.dump(cache, f, protocol=pickle.HIGHEST_PROTOCOL)
    tmp_path.replace(path)  # atomic on same filesystem
    print(f"Cache saved: {len(cache):,} entries → {path}")


# Load only the key index (350MB) instead of the full 32GB cache.
# Run extract_cache_keys.py once in a terminal to generate this file.
KEYS_PATH = CACHE_PATH.with_name("embedding_cache_keys.pkl")
if KEYS_PATH.exists():
    with open(KEYS_PATH, "rb") as f:
        embedding_cache_keys = pickle.load(f)
    print(f"Key index loaded: {len(embedding_cache_keys):,} hashes ({KEYS_PATH.stat().st_size/1e6:.0f} MB)")
    embedding_cache = None  # full cache not loaded — using key index only
elif CACHE_PATH.exists():
    print("WARNING: Key index not found. Run extract_cache_keys.py first to avoid OOM crashes.")
    print("Falling back to loading full cache (may crash on machines with < 40GB RAM).")
    with open(CACHE_PATH, "rb") as f:
        embedding_cache = pickle.load(f)
    embedding_cache_keys = set(embedding_cache.keys())
    print(f"Cache loaded: {len(embedding_cache_keys):,} entries from {CACHE_PATH}")
else:
    embedding_cache = {}
    embedding_cache_keys = set()
    print(f"No cache found at {CACHE_PATH} — starting fresh.")

Cache loaded: 5,527,109 entries from /Users/stephanie.mcmahon/smcmahon_repo/auto_classification/artifacts/cache/embedding_cache.pkl


## 4. Embed L4 Anchor Descriptions

Only 13 embeddings — completes in seconds.

In [6]:
bedrock_client = get_bedrock_client(profile_name=AWS_PROFILE, region=AWS_REGION)

anchor_hashes = [stable_text_hash(t) for t in anchor_texts]

anchor_matrix = embed_texts_from_cache(
    texts=anchor_texts,
    text_hashes=anchor_hashes,
    cache=embedding_cache,
    client=bedrock_client,
    model_id=MODEL_ID,
    show_progress=True,
    max_workers=1,
    max_retries=MAX_RETRIES,
)

print(f"Anchor matrix shape: {anchor_matrix.shape}")

# L2-normalize anchors for cosine similarity via dot product
anchor_norms = np.linalg.norm(anchor_matrix, axis=1, keepdims=True)
anchor_matrix_normed = anchor_matrix / np.clip(anchor_norms, 1e-10, None)
print("Anchor embeddings normalised.")

# Save cache so anchor embeddings are persisted
save_cache(embedding_cache, CACHE_PATH)

Anchor matrix shape: (13, 1536)
Anchor embeddings normalised.
Cache saved: 5,527,109 entries → /Users/stephanie.mcmahon/smcmahon_repo/auto_classification/artifacts/cache/embedding_cache.pkl


## 5. Build Listing Texts and Compute Embeddings

For the validate_products run these will already be cached (LEI embedding run).
For classify_services, most will be missing and require the overnight embedding job.
Set `RUN_EMBEDDING = False` in config to do a dry-run that reports cache coverage only.

In [7]:
texts = build_product_text(df).tolist()
hashes = [stable_text_hash(t) for t in texts]

missing_hashes = [h for h in set(hashes) if h not in embedding_cache]
cache_hits = len(set(hashes)) - len(missing_hashes)
cache_pct = cache_hits / len(set(hashes)) * 100 if hashes else 0

print(f"Total listings:         {len(df):,}")
print(f"Unique text hashes:     {len(set(hashes)):,}")
print(f"Cache hits:             {cache_hits:,} ({cache_pct:.1f}%)")
print(f"Missing (need embed):   {len(missing_hashes):,}")

if missing_hashes and not RUN_EMBEDDING:
    print("\nRUN_EMBEDDING=False — stopping here. Set RUN_EMBEDDING=True to embed missing texts.")
    print("Run this as an overnight job for large missing counts.")

Total listings:         7,162,087
Unique text hashes:     6,768,580
Cache hits:             5,000,002 (73.9%)
Missing (need embed):   1,768,578


In [8]:
# ── Incremental cache: new embeddings go to a SEPARATE small file ─────────
# The main embedding_cache (~32GB) is never rewritten mid-run.
# Only newly computed embeddings are checkpointed — fast, small writes.
NEW_CACHE_PATH = CACHE_PATH.with_name("embedding_cache_new.pkl")

if NEW_CACHE_PATH.exists():
    with open(NEW_CACHE_PATH, "rb") as f:
        new_embeddings = pickle.load(f)
    print(f"Resuming: {len(new_embeddings):,} new embeddings already in incremental cache")
else:
    new_embeddings = {}
    print("Starting fresh incremental cache")

class _MergedCache(dict):
    """Dict that checks base key index before reporting a miss. Writes go here only."""
    def __init__(self, base_keys, new_entries):
        super().__init__(new_entries)
        self._base_keys = base_keys  # set of hashes — no values loaded
    def __contains__(self, key):
        return super().__contains__(key) or key in self._base_keys
    def __getitem__(self, key):
        if super().__contains__(key):
            return super().__getitem__(key)
        # Value not in new_embeddings — will be looked up from full cache at classification time
        raise KeyError(f"Hash {key[:8]}... is in base cache keys but values not loaded (by design)")

merged_cache = _MergedCache(embedding_cache_keys, new_embeddings)

# Recompute missing using merged view
missing_hashes_merged = [h for h in set(hashes) if h not in merged_cache]
print(f"Missing after merging incremental cache: {len(missing_hashes_merged):,}")

if missing_hashes_merged and not RUN_EMBEDDING:
    print("RUN_EMBEDDING=False — skipping embed step.")
elif missing_hashes_merged:
    def on_checkpoint(cache, processed):
        new_only = dict(cache)
        tmp = NEW_CACHE_PATH.with_suffix('.pkl.tmp')
        with open(tmp, 'wb') as f:
            pickle.dump(new_only, f, protocol=pickle.HIGHEST_PROTOCOL)
        tmp.replace(NEW_CACHE_PATH)
        print(f"Checkpointed {len(new_only):,} new embeddings → {NEW_CACHE_PATH.name}")

    # Pass only missing texts — avoids building a huge return matrix for cached items
    text_by_hash = {stable_text_hash(t): t for t in texts}
    missing_texts_only = [text_by_hash[h] for h in missing_hashes_merged]

    embed_texts_from_cache(
        texts=missing_texts_only,
        text_hashes=missing_hashes_merged,
        cache=merged_cache,
        client=bedrock_client,
        model_id=MODEL_ID,
        show_progress=True,
        max_workers=MAX_WORKERS,
        checkpoint_every=50000,
        on_checkpoint=on_checkpoint,
        max_retries=MAX_RETRIES,
    )
    del missing_texts_only  # free memory immediately

    new_final = dict(merged_cache)
    save_cache(new_final, NEW_CACHE_PATH)
    print(f"Embedding complete. Volume 2: {len(new_final):,} new embeddings saved to {NEW_CACHE_PATH.name}")
    print("Run the classification script (classify_services_batched.py) to produce L4 assignments.")
else:
    print("All embeddings already in cache — no new Bedrock calls needed.")
    print("Run the classification script (classify_services_batched.py) to produce L4 assignments.")

listing_matrix_normed = None  # Classification is done via batched script, not in-memory matrix

Starting fresh incremental cache
Missing after merging incremental cache: 1,768,578


Embedding missing texts:   3%|▎         | 50021/1768578 [24:07<11:49:21, 40.38it/s]

Checkpointed 50,000 new embeddings → embedding_cache_new.pkl


Embedding missing texts:   6%|▌         | 100029/1768578 [49:07<11:01:02, 42.07it/s]

Checkpointed 100,000 new embeddings → embedding_cache_new.pkl


Embedding missing texts:   7%|▋         | 123865/1768578 [1:01:20<20:32:26, 22.24it/s] 

: 

## 6. Cosine Similarity Classification

Similarity matrix: `(N listings × 1536) @ (1536 × 13 anchors)` — fast matrix multiply.
Each listing is assigned to the anchor with the highest cosine similarity.

In [ ]:
assert listing_matrix_normed is not None, "Run embedding step first (RUN_EMBEDDING=True)."

# (N, 13) similarity matrix
similarity_matrix = listing_matrix_normed @ anchor_matrix_normed.T
print(f"Similarity matrix shape: {similarity_matrix.shape}")

assigned_idx    = similarity_matrix.argmax(axis=1)
confidence      = similarity_matrix.max(axis=1)
second_best_idx = np.argsort(similarity_matrix, axis=1)[:, -2]
second_best_score = similarity_matrix[np.arange(len(df)), second_best_idx]
margin          = confidence - second_best_score  # gap between top-1 and top-2

df["ASSIGNED_L4_IDX"]       = assigned_idx
df["ASSIGNED_L4_ID"]        = [anchor_ids[i]    for i in assigned_idx]
df["ASSIGNED_L4_LABEL"]     = [anchor_labels[i] for i in assigned_idx]
df["CONFIDENCE"]            = confidence.round(4)
df["CONFIDENCE_MARGIN"]     = margin.round(4)
df["IS_LOW_CONFIDENCE"]     = confidence < CONFIDENCE_THRESHOLD

n_low = df["IS_LOW_CONFIDENCE"].sum()
print(f"\nThreshold: {CONFIDENCE_THRESHOLD}")
print(f"High-confidence assignments: {len(df) - n_low:,} ({(len(df)-n_low)/len(df)*100:.1f}%)")
print(f"Low-confidence (residual):   {n_low:,} ({n_low/len(df)*100:.1f}%)")

## 7. Inspect Assignment Distribution

In [ ]:
dist = (
    df.groupby(["ASSIGNED_L4_LABEL", "IS_LOW_CONFIDENCE"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={False: "HIGH_CONF", True: "LOW_CONF"})
)
dist["TOTAL"] = dist.sum(axis=1)
dist["PCT"]   = (dist["TOTAL"] / len(df) * 100).round(1)
dist = dist.sort_values("TOTAL", ascending=False)

print("L4 Assignment Distribution:")
print(dist.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
labels = dist.index.tolist()
x = range(len(labels))
high = dist["HIGH_CONF"] if "HIGH_CONF" in dist.columns else pd.Series(0, index=dist.index)
low  = dist["LOW_CONF"]  if "LOW_CONF"  in dist.columns else pd.Series(0, index=dist.index)
ax.bar(x, high, label="High confidence", color="steelblue")
ax.bar(x, low, bottom=high, label="Low confidence", color="tomato")
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Listings")
ax.set_title(f"L4 Assignment Distribution — {MODE} (threshold={CONFIDENCE_THRESHOLD})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confidence score distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["CONFIDENCE"], bins=100, color="steelblue", edgecolor="none")
ax.axvline(CONFIDENCE_THRESHOLD, color="tomato", linestyle="--", label=f"Threshold ({CONFIDENCE_THRESHOLD})")
ax.set_xlabel("Cosine Similarity (Top-1)")
ax.set_ylabel("Listings")
ax.set_title("Confidence Score Distribution")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nConfidence percentiles:")
for pct in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{pct:2d}: {np.percentile(df['CONFIDENCE'], pct):.4f}")

## 8. Validation Check (validate_products mode)

Spot-check assignment quality. For known product types, verify they land on
expected L4s. If common product types are landing in wrong buckets, adjust
anchor descriptions in `l4_taxonomy_anchors.json` and re-run.

In [ ]:
if MODE == "validate_products" and "CURRENT_L3" in df.columns:
    print("Current L3 → Assigned L4 cross-tab (counts):")
    cross = pd.crosstab(
        df["CURRENT_L3"],
        df["ASSIGNED_L4_LABEL"],
        margins=True,
    )
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(cross)
else:
    print("Validation cross-tab only runs in validate_products mode.")

In [ ]:
# Sample 10 high-confidence and 10 low-confidence listings for manual review
print("=== 10 HIGH-CONFIDENCE SAMPLES ===")
high_sample = df[~df["IS_LOW_CONFIDENCE"]].sample(min(10, (~df["IS_LOW_CONFIDENCE"]).sum()), random_state=42)
for _, row in high_sample.iterrows():
    name = str(row.get("PRODUCT_NAME", ""))[:80]
    print(f"  [{row['CONFIDENCE']:.3f}] {row['ASSIGNED_L4_LABEL']:<35} | {name}")

print("\n=== 10 LOW-CONFIDENCE SAMPLES ===")
if df["IS_LOW_CONFIDENCE"].any():
    low_sample = df[df["IS_LOW_CONFIDENCE"]].sample(min(10, df["IS_LOW_CONFIDENCE"].sum()), random_state=42)
    for _, row in low_sample.iterrows():
        name = str(row.get("PRODUCT_NAME", ""))[:80]
        print(f"  [{row['CONFIDENCE']:.3f}] {row['ASSIGNED_L4_LABEL']:<35} | {name}")
else:
    print("  No low-confidence items at this threshold.")

## 9. Residual Analysis

Low-confidence listings are candidates for residual clustering to discover
missing L4 categories. Inspect before sending to the clustering notebook.

In [ ]:
residual = df[df["IS_LOW_CONFIDENCE"]].copy()
print(f"Residual size: {len(residual):,} listings ({len(residual)/len(df)*100:.1f}% of total)")

if len(residual) > 0:
    print("\nTop L4s assigned to residual (where assignment was uncertain):")
    print(residual["ASSIGNED_L4_LABEL"].value_counts().head(10).to_string())

    if "CURRENT_L3" in residual.columns:
        print("\nCurrent L3 breakdown of residual:")
        print(residual["CURRENT_L3"].value_counts().head(10).to_string())

## 10. Save Results

In [ ]:
output_dir = LOCAL_OUTPUT_DIR / RUN_TAG
output_dir.mkdir(parents=True, exist_ok=True)

# Save full classification results
results_path = output_dir / "classification_results.csv"
output_cols = ["PRODUCT_ID", "ASSIGNED_L4_ID", "ASSIGNED_L4_LABEL", "CONFIDENCE", "CONFIDENCE_MARGIN", "IS_LOW_CONFIDENCE"]
if "CURRENT_L3" in df.columns:
    output_cols.insert(2, "CURRENT_L3")
if "CURRENT_L4" in df.columns:
    output_cols.insert(3, "CURRENT_L4")

df[output_cols].to_csv(results_path, index=False)
print(f"Results saved: {results_path}")

# Save residual separately for clustering notebook
if len(residual) > 0:
    residual_path = output_dir / "residual_for_clustering.csv"
    residual[output_cols].to_csv(residual_path, index=False)
    print(f"Residual saved: {residual_path} ({len(residual):,} rows)")

# Save run summary
summary = {
    "mode": MODE,
    "total_listings": len(df),
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "high_confidence_count": int((~df["IS_LOW_CONFIDENCE"]).sum()),
    "low_confidence_count": int(df["IS_LOW_CONFIDENCE"].sum()),
    "l4_distribution": df["ASSIGNED_L4_LABEL"].value_counts().to_dict(),
    "confidence_percentiles": {
        str(p): float(np.percentile(df["CONFIDENCE"], p))
        for p in [10, 25, 50, 75, 90, 95, 99]
    },
}
with open(output_dir / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved: {output_dir / 'run_summary.json'}")

In [ ]:
if SAVE_TO_SNOWFLAKE:
    from snowflake.snowpark import functions as F

    out_df = df[["PRODUCT_ID", "ASSIGNED_L4_ID", "ASSIGNED_L4_LABEL", "CONFIDENCE", "IS_LOW_CONFIDENCE"]].copy()
    out_df.columns = [c.upper() for c in out_df.columns]

    sp_df = sf_session.create_dataframe(out_df)
    sp_df.write.mode("overwrite").save_as_table(OUTPUT_TABLE)
    print(f"Written {len(out_df):,} rows to {OUTPUT_TABLE}")
else:
    print("Snowflake write skipped (SAVE_TO_SNOWFLAKE=False).")